# step4 — 지침 관측 (RQ3)

**무엇을 확인하나:** 모델이 새 함수 이름을 쓰려는 순간, **지침 속 표기 지시어 단어**("camelCase")를
**얼마나 보는지(어텐션)** 를 층별로 잰다. 코드 이름 어텐션과 비교해서, 지시어를 보긴 보는지·어느 층에서
보는지 확인한다.

**고정 설정:** 4모델 · 함수이름 504개(42묶음) · 무작위값 42 · 문맥은 규약6+위반6 · 지침 방향 2종(camel/snake).

> 코드 이름과 달리 **지시어는 프롬프트에 하나**뿐이라, 묶음 하나당 지시어 관측 1개. 42묶음 × 방향2 = 84개/모델.

> **메모리 주의(T4):** 어텐션을 다 꺼내서 무겁다. **deepseek-6.7b는 OOM 가능** → 셀 ④ 8bit 주석 해제.

**모델 하나씩.** 셀 ④ `PICK`에서 하나 고르고 셀 순서대로. 끊겨도 저장된 건 건너뜀. 마지막에 zip.


In [ ]:
# 환경 설정 — 설치, GPU 확인, 무작위값 42
!pip install -q transformers accelerate torch matplotlib pandas bitsandbytes
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)


In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# 조건 설정 — 4모델 중 하나, 42묶음 x 지침 방향 2종
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
# deepseek-6.7b OOM나면 아래 해제(8bit):
# if MODEL.family=='deepseek': MODEL = ModelSpec(name=MODEL.name, family='deepseek', dtype='float16', quantization='8bit')

DIRECTIONS = [Notation.CAMEL, Notation.SNAKE]   # 지침이 camel / snake
BLOCKS = list(range(42))

def obs_cond(target, block):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL, pool_block=block),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target), seed=42)

conditions = [obs_cond(t, b) for t in DIRECTIONS for b in BLOCKS]
print('모델:', MODEL.family, '| 조건 수:', len(conditions), '(=방향2 x 묶음42)')
print('예:', conditions[0].slug())


In [ ]:
# 실행 — 지침 관측. 조건마다 저장(재개). 어텐션 관측이라 eager.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np

STEP = 'step4_instr-observe'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')
if todo:
    handle = load_model(MODEL, attn_implementation='eager')
    print(f'  층수 {handle.num_layers} | GQA {handle.gqa_info()}')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ3'))
        if i % 10 == 0 or i == len(todo):
            pl = out.metrics.per_layer
            di = np.mean([v.get('instr_target_word__attention_weight',0) for v in pl.values()])
            cc = np.mean([v.get('code_camel__attention_weight',0) for v in pl.values()])
            print(f'    [{i}/{len(todo)}] 평균 어텐션  지시어 {di:.4f}  vs  코드이름 {cc:.4f}')
    print('  완료.')
else:
    print('  이미 다 됨')


In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step4_instr-observe')) for c in conditions
        if result_path(c, step='step4_instr-observe').exists()]
print('불러온 조건:', len(recs), '-> results/step4_instr-observe/')


In [ ]:
STEPS = ['step4_instr-observe']
# ⑧ 결과 zip으로 묶어 내려받기
import shutil, os, glob

def pack(step):
    d = f'results/{step}'
    if not os.path.isdir(d):
        print(f'  [건너뜀] {d} 폴더가 없다 — 이 스텝은 아직 안 돌렸다')
        return None
    n = len(glob.glob(f'{d}/*.json'))
    if n == 0:
        print(f'  [건너뜀] {d} 가 비어 있다')
        return None
    path = shutil.make_archive(step, 'zip', d)
    print(f'  {step}: {n}개 → {path} ({os.path.getsize(path)/1e6:.1f}MB)')
    return path

print('results/ 안에 있는 폴더:', sorted(os.listdir('results')) if os.path.isdir('results') else '(results 폴더 없음)')
print()
made = [p for p in (pack(s) for s in STEPS) if p]

if not made:
    print('\n내려받을 것이 없다. 실행 셀을 먼저 돌렸는지 확인할 것.')
else:
    try:
        from google.colab import files
        for p in made:
            files.download(p)
        print('\n다운로드 시작. 브라우저가 막으면 왼쪽 **파일 탐색기**에서 직접 받으면 된다.')
    except Exception as e:
        print(f'\nColab 자동 다운로드 불가({type(e).__name__}). 위 경로에서 직접 받을 것.')
